In [2]:
#!/usr/bin/env python3
"""
GERADOR — BASE DE DADOS DA AVALIAÇÃO
Curso: Big Data para Negócios | Avaliação Final (baseada no Lab 12)

Diferente do dataset do curso (transactions_synthetic.csv):
- Novo seed (123) — valores não batem com o que o aluno já viu
- 2 colunas novas: channel (app/web/pos/atm) e merchant_category
- Fraude correlacionada com CHANNEL + MERCHANT_CATEGORY, não só segmento
  (obriga o aluno a explorar de novo, não reciclar a lógica do Lab 12)
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

N = 30_000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 12, 31)

print("Gerando base de dados da avaliação...")

# ----------------------------------------------------------------------
# Segmento e score (mesma lógica conceitual do curso, valores diferentes)
# ----------------------------------------------------------------------
segments = np.random.choice(['Premium', 'Standard', 'High-Risk'], size=N, p=[0.55, 0.35, 0.10])
credit_score = np.clip(np.random.normal(640, 140, N).astype(int), 300, 900)

# ----------------------------------------------------------------------
# Novas dimensões — não existiam no dataset do curso
# ----------------------------------------------------------------------
channel = np.random.choice(['app', 'web', 'pos', 'atm'], size=N, p=[0.40, 0.30, 0.20, 0.10])
merchant_category = np.random.choice(
    ['varejo', 'viagem', 'eletronico', 'alimentacao', 'servicos', 'saude'],
    size=N, p=[0.30, 0.10, 0.15, 0.20, 0.15, 0.10]
)
transaction_type = np.random.choice(['compra', 'saque', 'transferencia', 'pagamento'], size=N, p=[0.55, 0.15, 0.20, 0.10])

# ----------------------------------------------------------------------
# Valores e datas
# ----------------------------------------------------------------------
amount = np.clip(np.random.lognormal(4.3, 1.3, N), 5, 40_000)
dates = [START_DATE + timedelta(days=random.randint(0, (END_DATE-START_DATE).days),
                                  hours=random.randint(0, 23)) for _ in range(N)]
risk_score = np.random.uniform(0, 100, N)

# ----------------------------------------------------------------------
# Fraude: NOVA lógica — depende de channel + merchant_category + hora
# (padrão real: fraude em apps costuma concentrar em compras de viagem
#  feitas de madrugada — o aluno precisa DESCOBRIR isso na EDA)
# ----------------------------------------------------------------------
is_fraud = []
for i in range(N):
    base = {'Premium': 0.004, 'Standard': 0.012, 'High-Risk': 0.045}[segments[i]]
    if channel[i] == 'app':
        base *= 2.2
    if merchant_category[i] == 'viagem':
        base *= 2.8
    if dates[i].hour < 5:  # madrugada
        base *= 1.8
    if risk_score[i] > 80:
        base *= 1.5
    is_fraud.append(random.random() < min(base, 0.35))

status = ['declined' if f or random.random() < 0.08 else 'approved' for f in is_fraud]

df = pd.DataFrame({
    'transaction_id': range(1, N+1),
    'customer_id': np.random.randint(1, 8000, N),
    'amount': amount.round(2),
    'transaction_type': transaction_type,
    'channel': channel,
    'merchant_category': merchant_category,
    'timestamp': dates,
    'status': status,
    'risk_score': risk_score.round(2),
    'segment': segments,
    'credit_score': credit_score,
    'is_fraud': is_fraud,
})
df = df.sort_values('timestamp').reset_index(drop=True)

df.to_csv('/content/avaliacao_transactions.csv', index=False)

print(f"✓ {len(df)} linhas geradas")
print(f"  Taxa de fraude geral: {df['is_fraud'].mean():.2%}")
print(f"  Fraude por channel:")
print(df.groupby('channel')['is_fraud'].mean().sort_values(ascending=False).apply(lambda x: f"{x:.2%}"))
print(f"  Fraude por merchant_category:")
print(df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=False).apply(lambda x: f"{x:.2%}"))
print(f"  Fraude por segmento:")
print(df.groupby('segment')['is_fraud'].mean().sort_values(ascending=False).apply(lambda x: f"{x:.2%}"))


Gerando base de dados da avaliação...
✓ 30000 linhas geradas
  Taxa de fraude geral: 2.50%
  Fraude por channel:
channel
app    3.79%
atm    1.80%
web    1.74%
pos    1.42%
Name: is_fraud, dtype: object
  Fraude por merchant_category:
merchant_category
viagem         5.32%
saude          2.47%
alimentacao    2.31%
varejo         2.24%
servicos       1.97%
eletronico     1.96%
Name: is_fraud, dtype: object
  Fraude por segmento:
segment
High-Risk    9.67%
Standard     2.93%
Premium      0.95%
Name: is_fraud, dtype: object


In [3]:
# ============================================================
# PIPELINE — Avaliação Big Data para Negócios (TechPay)
# Rota sem cluster: DuckDB + Parquet
# Cole cada bloco numa célula separada do Colab, na ordem.
# ============================================================

# --- Célula 1: instalar e importar ---
!pip install duckdb --quiet
import duckdb
import pandas as pd

con = duckdb.connect('techpay.duckdb')

# ============================================================
# ETAPA A — INGESTÃO
# Trazer o CSV gerado pelo script do professor para o ambiente.
# ============================================================

# --- Célula 2: ingestão ---
# Ajuste o caminho se o seu CSV estiver em outro lugar do Colab
csv_path = '/content/avaliacao_transactions.csv'

con.execute(f"""
    CREATE OR REPLACE TABLE raw_transactions AS
    SELECT * FROM read_csv_auto('{csv_path}')
""")

n_raw = con.execute("SELECT COUNT(*) FROM raw_transactions").fetchone()[0]
print(f"Linhas na tabela raw: {n_raw}")
con.execute("DESCRIBE raw_transactions").df()  # confira os tipos inferidos

# ============================================================
# ETAPA B — TABELA RAW COM SCHEMA DEFINIDO
# read_csv_auto infere tipos, mas o enunciado pede schema explícito.
# ============================================================

# --- Célula 3: tabela raw com tipos explícitos ---
con.execute("""
    CREATE OR REPLACE TABLE raw_typed AS
    SELECT
        CAST(transaction_id AS BIGINT)      AS transaction_id,
        CAST(customer_id AS BIGINT)         AS customer_id,
        CAST(amount AS DOUBLE)              AS amount,
        CAST(transaction_type AS VARCHAR)   AS transaction_type,
        CAST(channel AS VARCHAR)            AS channel,
        CAST(merchant_category AS VARCHAR)  AS merchant_category,
        CAST(timestamp AS TIMESTAMP)        AS timestamp,
        CAST(status AS VARCHAR)             AS status,
        CAST(risk_score AS DOUBLE)          AS risk_score,
        CAST(segment AS VARCHAR)            AS segment,
        CAST(credit_score AS INTEGER)       AS credit_score,
        CAST(is_fraud AS BOOLEAN)           AS is_fraud
    FROM raw_transactions
""")
print("Schema aplicado.")

# ============================================================
# ETAPA C — PARTICIONAMENTO
# Particiona por mês do timestamp, conforme decidido na Etapa 1.
# ============================================================

# --- Célula 4: particionamento (grava em Parquet particionado) ---
con.execute("""
    COPY (
        SELECT *, strftime(timestamp, '%Y-%m') AS particao_mes
        FROM raw_typed
    ) TO '/content/dados_particionados' (FORMAT PARQUET, PARTITION_BY particao_mes, OVERWRITE_OR_IGNORE)
""")
print("Dados particionados por mês em /content/dados_particionados")

# ============================================================
# ETAPA D — BRONZE
# Limpeza: duplicatas, valores impossíveis, tipos inconsistentes.
# ============================================================

# --- Célula 5: bronze ---
con.execute("""
    CREATE OR REPLACE TABLE bronze AS
    SELECT DISTINCT *
    FROM raw_typed
    WHERE amount > 0                     -- valor negativo/zero não é transação real
      AND credit_score BETWEEN 300 AND 900  -- fora da faixa = erro de captura
      AND risk_score BETWEEN 0 AND 100
      AND customer_id IS NOT NULL
      AND timestamp IS NOT NULL
""")

n_bronze = con.execute("SELECT COUNT(*) FROM bronze").fetchone()[0]
print(f"Linhas na Bronze: {n_bronze} (descartadas: {n_raw - n_bronze})")

# ============================================================
# ETAPA E — SILVER
# Enriquecimento: colunas derivadas que ajudam a análise depois.
# ============================================================

# --- Célula 6: silver ---
con.execute("""
    CREATE OR REPLACE TABLE silver AS
    SELECT
        *,
        CASE
            WHEN amount < 100 THEN 'baixo'
            WHEN amount < 1000 THEN 'medio'
            ELSE 'alto'
        END AS faixa_valor,
        CASE
            WHEN EXTRACT(HOUR FROM timestamp) < 6 THEN 'madrugada'
            WHEN EXTRACT(HOUR FROM timestamp) < 12 THEN 'manha'
            WHEN EXTRACT(HOUR FROM timestamp) < 18 THEN 'tarde'
            ELSE 'noite'
        END AS periodo_dia,
        dayname(timestamp) AS dia_semana
    FROM bronze
""")
n_silver = con.execute("SELECT COUNT(*) FROM silver").fetchone()[0]
print(f"Linhas na Silver: {n_silver}")
print("Colunas channel e merchant_category mantidas — ver justificativa no relatório.")

# ============================================================
# ETAPA F — GOLD
# Pelo menos 2 tabelas agregadas, pensando no que a Etapa 3 precisa.
# ============================================================

# --- Célula 7: gold — tabela 1 (risco por canal e categoria) ---
con.execute("""
    CREATE OR REPLACE TABLE gold_risco_canal_categoria AS
    SELECT
        channel,
        merchant_category,
        periodo_dia,
        COUNT(*) AS total_transacoes,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS total_fraudes,
        ROUND(AVG(CASE WHEN is_fraud THEN 1.0 ELSE 0.0 END) * 100, 2) AS taxa_fraude_pct,
        ROUND(AVG(amount), 2) AS ticket_medio
    FROM silver
    GROUP BY channel, merchant_category, periodo_dia
    ORDER BY taxa_fraude_pct DESC
""")

# --- Célula 8: gold — tabela 2 (resumo diário por segmento) ---
con.execute("""
    CREATE OR REPLACE TABLE gold_resumo_diario_segmento AS
    SELECT
        CAST(timestamp AS DATE) AS data,
        segment,
        COUNT(*) AS total_transacoes,
        SUM(amount) AS volume_total,
        ROUND(AVG(risk_score), 2) AS risk_score_medio,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS total_fraudes
    FROM silver
    GROUP BY CAST(timestamp AS DATE), segment
    ORDER BY data
""")

print("Tabelas Gold criadas: gold_risco_canal_categoria, gold_resumo_diario_segmento")
con.execute("SELECT * FROM gold_risco_canal_categoria LIMIT 10").df()

# ============================================================
# Exportar as tabelas Gold pra usar na Etapa 3 sem reprocessar tudo
# ============================================================

# --- Célula 9: exportar Gold ---
con.execute("COPY gold_risco_canal_categoria TO '/content/gold_risco_canal_categoria.parquet' (FORMAT PARQUET)")
con.execute("COPY gold_resumo_diario_segmento TO '/content/gold_resumo_diario_segmento.parquet' (FORMAT PARQUET)")
print("Gold exportada em Parquet — pronta para a Etapa 3.")

Linhas na tabela raw: 30000
Schema aplicado.
Dados particionados por mês em /content/dados_particionados
Linhas na Bronze: 30000 (descartadas: 0)
Linhas na Silver: 30000
Colunas channel e merchant_category mantidas — ver justificativa no relatório.
Tabelas Gold criadas: gold_risco_canal_categoria, gold_resumo_diario_segmento
Gold exportada em Parquet — pronta para a Etapa 3.


In [4]:
# ============================================================
# ETAPA 3 — Análises e Dashboard (TechPay)
# Usa as tabelas Gold exportadas na Etapa 2.
# Cole cada bloco numa célula separada do Colab, na ordem.
# ============================================================

# --- Célula 1: instalar e importar ---
!pip install duckdb plotly --quiet
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots

con = duckdb.connect()

# --- Célula 2: carregar as tabelas Gold ---
gold_canal = con.execute("""
    SELECT * FROM read_parquet('/content/gold_risco_canal_categoria.parquet')
""").df()

gold_diario = con.execute("""
    SELECT * FROM read_parquet('/content/gold_resumo_diario_segmento.parquet')
""").df()

print(f"gold_canal: {len(gold_canal)} linhas | gold_diario: {len(gold_diario)} linhas")

# ============================================================
# ANÁLISE 1 — Risco por canal
# ============================================================

# --- Célula 3 ---
risco_por_canal = con.execute("""
    SELECT channel,
           SUM(total_transacoes) AS total_transacoes,
           SUM(total_fraudes) AS total_fraudes,
           ROUND(SUM(total_fraudes) * 100.0 / SUM(total_transacoes), 2) AS taxa_fraude_pct
    FROM gold_canal
    GROUP BY channel
    ORDER BY taxa_fraude_pct DESC
""").df()
print(risco_por_canal)

# ============================================================
# ANÁLISE 2 — Risco por categoria de estabelecimento
# ============================================================

# --- Célula 4 ---
risco_por_categoria = con.execute("""
    SELECT merchant_category,
           SUM(total_transacoes) AS total_transacoes,
           SUM(total_fraudes) AS total_fraudes,
           ROUND(SUM(total_fraudes) * 100.0 / SUM(total_transacoes), 2) AS taxa_fraude_pct
    FROM gold_canal
    GROUP BY merchant_category
    ORDER BY taxa_fraude_pct DESC
""").df()
print(risco_por_categoria)

# ============================================================
# ANÁLISE 3 — Padrão temporal (período do dia)
# ============================================================

# --- Célula 5 ---
risco_por_periodo = con.execute("""
    SELECT periodo_dia,
           SUM(total_transacoes) AS total_transacoes,
           SUM(total_fraudes) AS total_fraudes,
           ROUND(SUM(total_fraudes) * 100.0 / SUM(total_transacoes), 2) AS taxa_fraude_pct
    FROM gold_canal
    GROUP BY periodo_dia
    ORDER BY taxa_fraude_pct DESC
""").df()
print(risco_por_periodo)

# --- Célula 6: tendência diária (para o bloco de tendência do dashboard) ---
tendencia_diaria = con.execute("""
    SELECT data,
           SUM(total_transacoes) AS total_transacoes,
           SUM(total_fraudes) AS total_fraudes,
           ROUND(SUM(total_fraudes) * 100.0 / SUM(total_transacoes), 2) AS taxa_fraude_pct
    FROM gold_diario
    GROUP BY data
    ORDER BY data
""").df()

# ============================================================
# KPIs gerais (bloco 1 do dashboard)
# ============================================================

# --- Célula 7 ---
total_transacoes = int(gold_canal['total_transacoes'].sum())
total_fraudes = int(gold_canal['total_fraudes'].sum())
taxa_geral = round(total_fraudes * 100.0 / total_transacoes, 2)
volume_total = float(gold_diario['volume_total'].sum())

print(f"Total transações: {total_transacoes}")
print(f"Total fraudes: {total_fraudes}")
print(f"Taxa geral de fraude: {taxa_geral}%")
print(f"Volume total: R$ {volume_total:,.2f}")

# ============================================================
# DASHBOARD — 4 blocos: KPI, tendência, composição, detalhe
# ============================================================

# --- Célula 8: montar o dashboard ---
fig = make_subplots(
    rows=3, cols=2,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}],
        [{"type": "xy", "colspan": 2}, None],
        [{"type": "bar"}, {"type": "bar"}],
    ],
    row_heights=[0.2, 0.4, 0.4],
    subplot_titles=("", "", "Tendência diária — taxa de fraude (%)", "",
                     "Composição — risco por canal", "Detalhe — risco por categoria"),
    vertical_spacing=0.12
)

# Bloco 1: KPIs
fig.add_trace(go.Indicator(
    mode="number", value=total_transacoes,
    title={"text": "Total de transações"}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=taxa_geral, number={"suffix": "%"},
    title={"text": "Taxa geral de fraude"}
), row=1, col=2)

# Bloco 2: Tendência
fig.add_trace(go.Scatter(
    x=tendencia_diaria['data'], y=tendencia_diaria['taxa_fraude_pct'],
    mode='lines', name='Taxa de fraude (%)', line=dict(width=2)
), row=2, col=1)

# Bloco 3: Composição por canal
fig.add_trace(go.Bar(
    x=risco_por_canal['channel'], y=risco_por_canal['taxa_fraude_pct'],
    name='Canal', marker_color='indianred'
), row=3, col=1)

# Bloco 4: Detalhe por categoria
fig.add_trace(go.Bar(
    x=risco_por_categoria['merchant_category'], y=risco_por_categoria['taxa_fraude_pct'],
    name='Categoria', marker_color='steelblue'
), row=3, col=2)

fig.update_layout(
    title_text="TechPay — Painel de Risco e Fraude",
    height=800, showlegend=False
)

fig.write_html('/content/dashboard_fraude.html')
print("Dashboard salvo em /content/dashboard_fraude.html")
fig.show()

gold_canal: 96 linhas | gold_diario: 1095 linhas
  channel  total_transacoes  total_fraudes  taxa_fraude_pct
0     app           12021.0          455.0             3.79
1     atm            3007.0           54.0             1.80
2     web            9004.0          157.0             1.74
3     pos            5968.0           85.0             1.42
  merchant_category  total_transacoes  total_fraudes  taxa_fraude_pct
0            viagem            3010.0          160.0             5.32
1             saude            2951.0           73.0             2.47
2       alimentacao            5976.0          138.0             2.31
3            varejo            9047.0          203.0             2.24
4          servicos            4576.0           90.0             1.97
5        eletronico            4440.0           87.0             1.96
  periodo_dia  total_transacoes  total_fraudes  taxa_fraude_pct
0   madrugada            7553.0          247.0             3.27
1       manha            7514.0  

In [5]:
# ============================================================
# ETAPA 4 (BÔNUS) — Modelo preditivo de fraude
# Regressão Logística + interpretação dos coeficientes
# Cole cada bloco numa célula separada do Colab, na ordem.
# ============================================================

# --- Célula 1: instalar e importar ---
!pip install duckdb scikit-learn --quiet
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

con = duckdb.connect()

# ============================================================
# Reconstrói a Silver (mesma lógica da Etapa 2) direto do CSV,
# pra ter dados linha a linha (a Gold já vem agregada, não serve pro modelo)
# ============================================================

# --- Célula 2: carregar e preparar os dados ---
csv_path = '/content/avaliacao_transactions.csv'

df = con.execute(f"""
    SELECT
        amount,
        transaction_type,
        channel,
        merchant_category,
        risk_score,
        segment,
        credit_score,
        CASE
            WHEN EXTRACT(HOUR FROM CAST(timestamp AS TIMESTAMP)) < 6 THEN 'madrugada'
            WHEN EXTRACT(HOUR FROM CAST(timestamp AS TIMESTAMP)) < 12 THEN 'manha'
            WHEN EXTRACT(HOUR FROM CAST(timestamp AS TIMESTAMP)) < 18 THEN 'tarde'
            ELSE 'noite'
        END AS periodo_dia,
        CAST(is_fraud AS INTEGER) AS is_fraud
    FROM read_csv_auto('{csv_path}')
    WHERE amount > 0 AND credit_score BETWEEN 300 AND 900 AND risk_score BETWEEN 0 AND 100
""").df()

print(f"Linhas para o modelo: {len(df)}")
print(f"Taxa de fraude na base: {df['is_fraud'].mean():.2%}")

# --- Célula 3: preparar features (one-hot para categóricas) ---
categoricas = ['transaction_type', 'channel', 'merchant_category', 'segment', 'periodo_dia']
numericas = ['amount', 'risk_score', 'credit_score']

X = pd.get_dummies(df[categoricas + numericas], columns=categoricas, drop_first=True)
y = df['is_fraud']

# Padronizar as numéricas ajuda a comparar os coeficientes entre si
scaler = StandardScaler()
X[numericas] = scaler.fit_transform(X[numericas])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")

# ============================================================
# Treinar o modelo
# ============================================================

# --- Célula 4: treinar ---
# class_weight='balanced' porque fraude é rara (~2.5%) — sem isso o modelo
# aprenderia a simplesmente prever "não é fraude" sempre e ainda acertaria bastante.
modelo = LogisticRegression(max_iter=1000, class_weight='balanced')
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['legítima', 'fraude']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.3f}")

# ============================================================
# Interpretar os coeficientes
# ============================================================

# --- Célula 5: coeficientes ordenados por impacto ---
coeficientes = pd.DataFrame({
    'variavel': X.columns,
    'coeficiente': modelo.coef_[0]
})
coeficientes['odds_ratio'] = np.exp(coeficientes['coeficiente'])
coeficientes = coeficientes.sort_values('coeficiente', ascending=False)

print(coeficientes.to_string(index=False))

Linhas para o modelo: 30000
Taxa de fraude na base: 2.50%
Treino: 22500 | Teste: 7500
              precision    recall  f1-score   support

    legítima       0.99      0.72      0.83      7312
      fraude       0.05      0.63      0.10       188

    accuracy                           0.72      7500
   macro avg       0.52      0.68      0.47      7500
weighted avg       0.96      0.72      0.82      7500

AUC-ROC: 0.741
                      variavel  coeficiente  odds_ratio
      merchant_category_viagem     1.019469    2.771723
transaction_type_transferencia     0.321150    1.378712
        transaction_type_saque     0.290326    1.336864
    transaction_type_pagamento     0.225223    1.252603
      merchant_category_varejo     0.139861    1.150114
                  credit_score     0.113276    1.119941
                    risk_score     0.107734    1.113751
       merchant_category_saude     0.035725    1.036371
                        amount    -0.031997    0.968510
    merchant